# MAS Lab Notebook - APF-Based Obstacle Avoidance and Formation Control

## 1. Project Overview & Objectives
The goal of this notebook is to design and evaluate a multi-agent system (MAS) capable of navigating an environment while maintaining a structured formation and avoiding static obstacles.

Specifically, we aim to:
1. **Model Agent Dynamics:** Implement a Single Integrator Dynamics system to control the motion of each agent.
2. **Establish Formation Control:** Coordinate a fleet of agents to follow a moving Leader while maintaining a defined target geometry.
3. **Implement Obstacle Avoidance:** Apply Artificial Potential Fields (APF) to prevent collisions between the agents and environmental obstacles.
4. **Enable Dynamic Formation Adaptation:** Introduce a formation-switching mechanism that allows the fleet to alter its shape dynamically to navigate narrow passages.

---

## 2. Methodology & Incremental Development Strategy
[WIP] Given the complexity of multi-agent control systems, we adopt an **incremental, step-by-step approach**. Each component will be developed, tested, and validated independently prior to system integration:

* **Step 1:** Single Integrator Dynamics & Target Following
* **Step 2:** Static Multi-Agent Formation Control (Leader-Follower framework)
* **Step 3:** Artificial Potential Field (APF) Integration for Obstacle Avoidance
* **Step 4:** Dynamic Formation Adaptation (Reconfiguration in constrained spaces)
* **Step 5:** Full Pipeline Integration and Performance Evaluation

---
## 3. Experimentation
### 3.1 Requirements [WIP]
I am using a conda environment with the following packages:
- numpy
- matplotlib
- shapely
- ..

In [27]:
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

### 3.2 The `Agent` Class

In [18]:
def normalize(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    if norm == 0.0:
        return np.zeros_like(v)
    return v / norm


@dataclass
class Agent:
    position: np.ndarray
    target: np.ndarray
    speed: float

    def __post_init__(self):
        self.position = np.array(self.position, dtype=float)
        self.target = np.array(self.target, dtype=float)

    def direction_to_target(self) -> np.ndarray:
        return normalize(self.target - self.position)

    def velocity(self) -> np.ndarray:
        return self.speed * self.direction_to_target()

    def step(self, dt: float) -> None:
        if self.reached_target(tol=self.speed * dt):
            self.position = self.target.copy()
            return
        self.position = self.position + dt * self.velocity()

    def distance_to_target(self) -> float:
        return np.linalg.norm(self.target - self.position)

    def reached_target(self, tol: float = 1e-2) -> bool:
        return self.distance_to_target() <= tol

In [ ]:
agent = Agent(position=[0.0, 0.0], target=[5.0, 3.0], speed=1.0)

def make_animation():
    dt = 0.05
    frames = 125

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.set_xlim(-0.5, 5.8)
    ax.set_ylim(-0.5, 3.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title('Single-integrator target following')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    ax.scatter(
        [agent.target[0]], [agent.target[1]],
        marker="*", s=220, color="crimson",
        edgecolors="black", zorder=4, label="target"
    )
    path_line, = ax.plot([], [], '--', color='tab:blue', lw=2.0, label='trajectory')
    agent_point, = ax.plot([], [], 'o', color='tab:blue', markersize=10, label='agent')
    time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')
    dist_text = ax.text(0.02, 0.92, '', transform=ax.transAxes, va='top')
    ax.legend(loc='lower right')

    xs = [agent.position[0]]
    ys = [agent.position[1]]

    def init():
        path_line.set_data(xs, ys)
        agent_point.set_data([agent.position[0]], [agent.position[1]])
        time_text.set_text('t = 0.00 s')
        dist_text.set_text(f'distance = {agent.distance_to_target():.2f}')
        return path_line, agent_point, time_text, dist_text

    def update(frame):
        agent.step(dt)
        xs.append(agent.position[0])
        ys.append(agent.position[1])
        path_line.set_data(xs, ys)
        agent_point.set_data([agent.position[0]], [agent.position[1]])
        time_text.set_text(f't = {(frame + 1) * dt:.2f} s')
        dist_text.set_text(f'distance = {agent.distance_to_target():.2f}')
        return path_line, agent_point, time_text, dist_text

    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=frames,
        interval=50,
        blit=True
        )

    plt.close(fig)
    html_animation = HTML(anim.to_jshtml())
    display(html_animation)


if __name__ == '__main__':
    make_animation()